Load Data – Reads the source file (NLP Lexicon - Cecielmotus.csv) into a Pandas DataFrame.

1. Select Relevant Columns – Retains only the essential fields: word, POS symbol, POS word, and meaning.
2. Handle Missing Words – Fills blank entries in the word column with the most recent non-empty value, ensuring that multiple part-of-speech usages of the same word are properly captured.
3. Normalize Text Values – Converts all text in the dataset to lowercase and trims extra spaces for uniform formatting.
4. Standardize Column Names – Renames columns by converting them to lowercase and replacing spaces with underscores (e.g., POS symbol → pos_symbol).
5. Export Cleaned Data – Saves the processed dataset as Cecielmotus_cleaned.csv for immediate use or download.

In [9]:
import pandas as pd
from google.colab import files  # <-- this makes 'files' available


df = pd.read_csv("NLP Lexicon - Cecielmotus.csv")
keep_cols = ["word", "POS symbol", "POS word", "meaning"]
df = df[[col for col in keep_cols if col in df.columns]]
df['word'] = df['word'].ffill()
for col in keep_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
df.columns = df.columns.str.lower().str.replace(" ", "_")
output_file = "Cecielmotus_cleaned.csv"
df.to_csv(output_file, index=False)
files.download(output_file)
print(f"Cleaning done. File saved and ready for download: {output_file}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaning done. File saved and ready for download: Cecielmotus_cleaned.csv


I want to make this code so I can add proper POS word and symbol to the 2nd sheets so I can ensure that they both have the same output so that the merging process would be easy to digest and no problems would arise later on

In [4]:
mapping_df = df.dropna(subset=["POS symbol", "POS word"])
mapping = mapping_df.drop_duplicates(subset=["POS symbol", "POS word"])

print("POS symbol → POS word mapping:\n")
for _, row in mapping.iterrows():
    print(f"{row['POS symbol']} → {row['POS word']}")

POS symbol → POS word mapping:

pt → particle
va → verbal affix
n → noun
intj → interjection
v → verb
adj → adjective
na → noun formative
pr → pronoun
adv → adverb
aa → adjective formative
con → conjunction
id → idiom
num → numeral
d → deictic


This script standardizes the second lexicon dataset to match the format of the first sheet, ensuring consistency across both.

1. Load Data – Reads the raw file (NLP Lexicon - Wikitionary.csv) into a Pandas DataFrame.
2. Normalize Column Names – Converts all column headers to lowercase and replaces spaces with underscores.
3. Rename Columns – Renames the part_of_speech column to pos_word for consistency.
4. Clean Text Values – Converts entries in word, pos_word, and meaning to lowercase and strips extra spaces.
5. Assign POS Symbols – Maps each pos_word (e.g., noun, verb, adjective) to its corresponding pos_symbol (e.g., n, v, adj) using a predefined dictionary. New categories like determiner, phrase, and phrasebook are also included in the mapping.
6. Reorder Columns – Arranges the dataset into the standardized format: word | pos_symbol | pos_word | meaning.
7. Export Cleaned File – Saves the processed dataset as Wikitionary_cleaned.csv and triggers an automatic download in Colab.

In [12]:
import pandas as pd
from google.colab import files
pos_map = {
    "particle": "pt",
    "verbal affix": "va",
    "noun": "n",
    "interjection": "intj",
    "verb": "v",
    "adjective": "adj",
    "noun formative": "na",
    "pronoun": "pr",
    "adverb": "adv",
    "adjective formative": "aa",
    "conjunction": "con",
    "idiom": "id",
    "numeral": "num",
    "deictic": "d",
    # new ones from your second sheet
    "determiner": "det",
    "phrase": "ph",
    "phrasebook": "pb"
}
df = pd.read_csv("NLP Lexicon - Wikitionary.csv")
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df = df.rename(columns={"part_of_speech": "pos_word"})
for col in ["word", "pos_word", "meaning"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
df["pos_symbol"] = df["pos_word"].map(pos_map)
df = df[["word", "pos_symbol", "pos_word", "meaning"]]
output_file = "Wikitionary_cleaned.csv"
df.to_csv(output_file, index=False)
files.download(output_file)

print(f"Cleaning done. File saved and ready for download: {output_file}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaning done. File saved and ready for download: Wikitionary_cleaned.csv


This script merges two cleaned lexicon datasets and ensures consistency by removing duplicates and blank entries. It also provides a detailed report of the cleaning process.

Steps performed:
1. Load Data - Reads the two cleaned CSV files (Cecielmotus_cleaned.csv and SecondSheet_cleaned.csv) into Pandas DataFrames.
2. Initial Report - Prints the number of rows and columns in each sheet before merging.
3. Merge Datasets - Concatenates both datasets into a single DataFrame for unified processing.
4. Remove Duplicates - Identifies and removes duplicate rows to avoid redundancy.
5. Reports - how many duplicates were found and dropped.
6. Handle Blank Meanings - Checks for rows where the meaning column is empty or missing (NaN or just spaces).
- Removes those rows
- Reports the total count of blank/NaN meanings removed.

7. Final Report
- Displays: Total rows before cleaning
- Duplicates removed
- Blank/NaN rows removed
- Total rows removed
- Final dataset shape (rows × columns)
- Also prints the count of missing values per column in the final dataset.

8. Save & Download
- Saves the cleaned, merged dataset as Lexicon_merged.csv.
- Automatically triggers a file download in Google Colab.

In [19]:
import pandas as pd
from google.colab import files
df1 = pd.read_csv("Cecielmotus_cleaned.csv")
df2 = pd.read_csv("Wikitionary_cleaned.csv")
print("=== Dataset Overview ===")
print(f"Sheet 1 (Cecielmotus_cleaned): {df1.shape[0]} rows, {df1.shape[1]} columns")
print(f"Sheet 2 (Wikitionary_cleaned): {df2.shape[0]} rows, {df2.shape[1]} columns")
merged_df = pd.concat([df1, df2], ignore_index=True)
total_before = merged_df.shape[0]
before_dupes = merged_df.shape[0]
merged_df = merged_df.drop_duplicates()
after_dupes = merged_df.shape[0]
duplicates_removed = before_dupes - after_dupes
before_blanks = merged_df.shape[0]
blank_meaning_mask = merged_df['meaning'].isna() | (merged_df['meaning'].astype(str).str.strip() == "")
blanks_count = blank_meaning_mask.sum()
merged_df = merged_df[~blank_meaning_mask]
after_blanks = merged_df.shape[0]
final_rows = merged_df.shape[0]
removed_total = total_before - final_rows

print("\n=== Merge Report ===")
print(f"Total rows combined (before cleaning): {total_before}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Rows with blank/NaN meaning removed: {blanks_count}")
print(f"Total rows removed: {removed_total}")
print(f"Final dataset shape: {final_rows} rows × {merged_df.shape[1]} columns")
missing = merged_df.isnull().sum()
print("\n=== Missing Values per Column (Final) ===")
print(missing)
output_file = "Lexicon_merged.csv"
merged_df.to_csv(output_file, index=False)
files.download(output_file)


=== Dataset Overview ===
Sheet 1 (Cecielmotus_cleaned): 4357 rows, 4 columns
Sheet 2 (Wikitionary_cleaned): 1987 rows, 4 columns

=== Merge Report ===
Total rows combined (before cleaning): 6344
Duplicates removed: 7
Rows with blank/NaN meaning removed: 15
Total rows removed: 22
Final dataset shape: 6322 rows × 4 columns

=== Missing Values per Column (Final) ===
word          0
pos_symbol    0
pos_word      0
meaning       0
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>